# Submission: гибридный поиск объявлений

Ноутбук строит `answer.csv` из корпуса объявлений и benchmark-запросов.

Кандидатов находят BM25, поиск вхождений запроса в заголовок, TF-IDF заголовков и эмбеддинги granite-embedding-97m-multilingual-r2. Для эмбеддингов используется GPU, если он доступен, иначе CPU. Логистический ранжировщик объединяет текстовую близость, семантическое сходство, локацию и признаки отзывов. Для знакомых запросов учитывается история выбора объявлений.

В конце ноутбук формирует до 50 объявлений на запрос и проверяет формат файла.
Ноутбук расчитан на работу на gpu, без него эмбеддинги будут считаться значительно дольше(на gpu фулл весь ноутбук выполняется за ~43 минуты), так же предполагается, что нужные файлы с датасетами лежат в одной директории с ноутбуком

### Обучение и проверка

Подход выбран на локальной валидации. Финальный ранжировщик обучается на доступных train-парах и кандидатах тех же поисковых каналов, которые используются для benchmark. Скрытые ответы benchmark при обучении и построении выдачи не используются.


In [1]:
# Импортируем библиотеки и задаём параметры поиска, обучения и сохранения ответа.

from pathlib import Path
from collections import Counter, defaultdict
from concurrent.futures import ThreadPoolExecutor
from itertools import chain
import re

import numpy as np
import pandas as pd
import tqdm
from scipy.sparse import csr_matrix, hstack, vstack
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer
from sklearn.linear_model import SGDClassifier
from sklearn.preprocessing import StandardScaler

DATA_DIR = Path.cwd()
ANSWER_PATH = DATA_DIR / "answer.csv"

QUERY_COLS = [
    "search_query", "search_location_id", "search_is_delivery_search",
    "search_infm_params_text", "search_category",
]

CANDIDATES_PER_CHANNEL = 200
USE_TITLE_TFIDF_CHANNEL = True  # title_rank модель требует этот канал и при inference
NEGATIVES_PER_CHANNEL = 40
PAIR_BATCH_SIZE = 25_000
SEARCH_BATCH_SIZE = 32
SEARCH_WORKERS = 2
RANDOM_SEED = 42


Читаем корпус, запросы и положительные пары, готовим идентификаторы для поиска

In [2]:
benchmark_items = pd.read_parquet(DATA_DIR / "benchmark_items.parquet")
benchmark_queries = pd.read_parquet(DATA_DIR / "benchmark_queries.parquet")
train_source = pd.read_parquet(
    DATA_DIR / "train.parquet",
    columns=QUERY_COLS + ["item_id"],
)

allowed_item_ids = frozenset(benchmark_items["item_id"])
item_row_by_id = pd.Series(
    np.arange(len(benchmark_items), dtype=np.int32),
    index=benchmark_items["item_id"],
)

# Удаляем повторные наблюдения одной и той же пары.
positive_pairs = train_source.drop_duplicates(QUERY_COLS + ["item_id"]).copy()
train_query_catalog = positive_pairs[QUERY_COLS].drop_duplicates().reset_index(drop=True)
train_query_catalog.insert(
    0, "query_id",
    [f"train_{i:07d}" for i in range(len(train_query_catalog))],
)
positive_pairs = positive_pairs.merge(
    train_query_catalog,
    on=QUERY_COLS,
    how="left",
    validate="many_to_one",
)

full_relevance = positive_pairs.groupby("query_id")["item_id"].agg(frozenset)
benchmark_relevance = full_relevance.map(
    lambda item_ids: item_ids & allowed_item_ids
)

train_queries = train_query_catalog.copy()
train_queries["relevant_items"] = train_queries["query_id"].map(benchmark_relevance)
train_queries = train_queries.loc[
    train_queries["relevant_items"].map(len).gt(0)
].reset_index(drop=True)

submission_queries = benchmark_queries.copy().reset_index(drop=True)

print(f"Корпус: {len(benchmark_items):,} объявлений")
print(f"Обучающих запросов: {len(train_queries):,}")
print(f"Benchmark-запросов: {len(submission_queries):,}")
print(f"Положительных пар для обучения: {train_queries['relevant_items'].map(len).sum():,}")


Корпус: 189,212 объявлений
Обучающих запросов: 26,556
Benchmark-запросов: 2,452
Положительных пар для обучения: 29,572


Очищаем и нормализуем тексты запросов и объявлений перед векторизацией

In [3]:
PARAM_LABELS = [
    "Вид услуги", "Тип услуги автосервиса", "Тип услуги",
    "Название услуги", "Специальность или сфера",
    "Предмет или специальность", "Место оказания услуг",
    "Тип стоимости", "Начальная цена",
]
PARAM_LABEL_PATTERN = r"(?<!\w)(?:" + "|".join(
    re.escape(label) for label in sorted(PARAM_LABELS, key=len, reverse=True)
) + r")(?!\w)"

DAY_VALUE = (
    r"(?i:понедельник|вторник|среда|четверг|пятница|суббота|воскресенье|"
    r"пн\.?|вт\.?|ср\.?|чт\.?|пт\.?|сб\.?|вс\.?)(?!\w)"
)
TECH_PARAM_PATTERNS = [
    r"График работы\s+(?:от|до)(?:\s+\d+(?::\d{2})?(?![\w:]))?",
    r"Время работы,?\s+(?:с|до)(?:\s+\d{1,2}:\d{2}(?![\w:]))?",
    r"(?:График работы,?\s+дни недели|Рабочие дни)(?:\s+" + DAY_VALUE + r")?",
    r"Признак предзаполнения прайс[ -]листа(?:\s+\d+(?!\w))?",
]
TECH_PARAM_PATTERN = r"(?<!\w)(?:" + "|".join(TECH_PARAM_PATTERNS) + r")(?!\w)"

PROMO_PHRASES = [
    "индивидуальный подход", "низкие цены",
    "добавляйте в избранное", "добавьте в избранное",
    "звоните", "пишите",
]
PROMO_PATTERN = r"(?i)(?<!\w)(?:" + "|".join(
    re.escape(phrase) for phrase in PROMO_PHRASES
) + r")(?!\w)"


def clean_params(series):
    return (
        series.fillna("")
        .str.replace(r"\s+", " ", regex=True)
        .str.replace(TECH_PARAM_PATTERN, " ", regex=True)
        .str.replace(PARAM_LABEL_PATTERN, " ", regex=True)
        .str.replace(r"\s+", " ", regex=True)
        .str.strip()
    )


def clean_description(series):
    return (
        series.fillna("")
        .str.replace(r"\s+", " ", regex=True)
        .str.replace(PROMO_PATTERN, " ", regex=True)
        .str.replace(r"\s+", " ", regex=True)
        .str.strip()
    )


def normalize_text(series):
    return (
        series.fillna("")
        .str.lower()
        .str.replace("ё", "е", regex=False)
        .str.replace(r"\s+", " ", regex=True)
        .str.strip()
    )


def normalize_for_title_match(series):
    return (
        series.fillna("")
        .str.lower()
        .str.replace("ё", "е", regex=False)
        .str.replace(r"[^0-9a-zа-я]+", " ", regex=True)
        .str.replace(r"\s+", " ", regex=True)
        .str.strip()
    )


def prepare_item_text(df):
    title = df["item_title_raw"].fillna("")
    params = clean_params(df["item_infm_params_text"])
    description = clean_description(df["item_description_raw"])
    return normalize_text(title + " " + title + " " + params + " " + description)


def prepare_query_text(df):
    query = df["search_query"].fillna("")
    filters = clean_params(df["search_infm_params_text"])
    return normalize_text(query + " " + filters)


Строим BM25-представление объявлений для первичного поиска текстовых кандидатов.

In [4]:
bm25_vectorizer = CountVectorizer(
    lowercase=True,
    ngram_range=(1, 1),
    min_df=2,
    max_df=0.98,
    max_features=200_000,
    dtype=np.float32,
)

item_count_matrix = bm25_vectorizer.fit_transform(
    tqdm.tqdm(
        prepare_item_text(benchmark_items),
        total=len(benchmark_items),
        desc="Построение BM25-корпуса",
    )
).tocsr()


def make_bm25_matrix(count_matrix, k1=1.5, b=0.75):
    count_matrix = count_matrix.tocsr().astype(np.float32)
    n_documents = count_matrix.shape[0]
    document_lengths = np.asarray(count_matrix.sum(axis=1)).ravel()
    average_document_length = document_lengths.mean()
    document_frequency = np.asarray(
        (count_matrix > 0).sum(axis=0)
    ).ravel()
    idf = np.log(
        1 + (n_documents - document_frequency + 0.5)
        / (document_frequency + 0.5)
    ).astype(np.float32)

    result = count_matrix.copy()
    row_indices = np.repeat(
        np.arange(n_documents), np.diff(result.indptr)
    )
    term_frequency = result.data
    length_normalization = (
        1 - b
        + b * document_lengths[row_indices] / average_document_length
    )
    result.data = (
        idf[result.indices]
        * term_frequency
        * (k1 + 1)
        / (term_frequency + k1 * length_normalization)
    ).astype(np.float32)
    return result


bm25_item_matrix = make_bm25_matrix(item_count_matrix, k1=1.5, b=0.75)
assert bm25_item_matrix.shape[0] == len(benchmark_items)
print("BM25-матрица:", bm25_item_matrix.shape)


Построение BM25-корпуса: 100%|██████████| 189212/189212 [00:27<00:00, 6961.60it/s]


BM25-матрица: (189212, 200000)


Индексируем заголовки и подготавливаем локацию, рейтинг и отзывы объявлений

In [5]:
normalized_titles = normalize_for_title_match(
    benchmark_items["item_title_raw"]
)
normalized_title_array = normalized_titles.to_numpy()
title_token_sets = [set(text.split()) for text in normalized_titles]

title_postings = {}
for item_row, title in enumerate(tqdm.tqdm(
    normalized_title_array,
    desc="Индекс заголовков",
)):
    for token in set(title.split()):
        title_postings.setdefault(token, []).append(item_row)
title_postings = {
    token: np.asarray(rows, dtype=np.int32)
    for token, rows in title_postings.items()
}

item_ids = benchmark_items["item_id"].to_numpy()
item_locations = pd.to_numeric(
    benchmark_items["item_location_id"], errors="coerce"
).to_numpy(dtype=float, na_value=np.nan)
item_ratings = pd.to_numeric(
    benchmark_items["item_rating"], errors="coerce"
).fillna(0).clip(0, 5).to_numpy(dtype=np.float32)
item_reviews = pd.to_numeric(
    benchmark_items["item_rating_reviews_count"], errors="coerce"
).fillna(0).clip(lower=0).to_numpy(dtype=np.float32)
item_quality = item_ratings * np.log1p(item_reviews)

print(f"Токенов в индексе заголовков: {len(title_postings):,}")


Индекс заголовков: 100%|██████████| 189212/189212 [00:00<00:00, 612950.89it/s]


Токенов в индексе заголовков: 42,482


Создаем функции для BM25 и для определения точных вхождений запроса в заголовки нужных объявлений

In [6]:
def get_topk_bm25(
    queries,
    query_matrix,
    top_k=CANDIDATES_PER_CHANNEL,
    batch_size=SEARCH_BATCH_SIZE,
    n_workers=SEARCH_WORKERS,
):
    top_k = min(top_k, len(benchmark_items))
    item_matrix_t = bm25_item_matrix.T.tocsc()
    starts = list(range(0, len(queries), batch_size))

    def process_batch(start):
        end = min(start + batch_size, len(queries))
        scores = (query_matrix[start:end] @ item_matrix_t).toarray()
        indices = np.argpartition(
            scores,
            kth=scores.shape[1] - top_k,
            axis=1,
        )[:, -top_k:]
        candidate_scores = np.take_along_axis(scores, indices, axis=1)
        order = np.argsort(candidate_scores, axis=1)[:, ::-1]
        return start, end, np.take_along_axis(indices, order, axis=1)

    predictions = {}
    query_ids = queries["query_id"].to_numpy()
    with ThreadPoolExecutor(max_workers=n_workers) as executor:
        results = executor.map(process_batch, starts)
        for start, end, indices in tqdm.tqdm(
            results,
            total=len(starts),
            desc="BM25 top-k",
        ):
            for local_row, query_row in enumerate(range(start, end)):
                predictions[query_ids[query_row]] = (
                    item_ids[indices[local_row]].tolist()
                )
    return predictions


def get_topk_exact_titles(
    queries,
    query_matrix,
    top_k=CANDIDATES_PER_CHANNEL,
):
    query_ids = queries["query_id"].to_numpy()
    query_texts = normalize_for_title_match(
        queries["search_query"]
    ).to_numpy()
    query_locations = pd.to_numeric(
        queries["search_location_id"], errors="coerce"
    ).to_numpy(dtype=float, na_value=np.nan)
    predictions = {}

    for query_row, (query_id, query) in enumerate(tqdm.tqdm(
        zip(query_ids, query_texts),
        total=len(queries),
        desc="Exact-title top-k",
    )):
        tokens = set(query.split())
        postings = [title_postings.get(token) for token in tokens]
        if not query or not postings or any(rows is None for rows in postings):
            predictions[query_id] = []
            continue

        base_rows = min(postings, key=len)
        padded_query = f" {query} "
        exact_rows = np.asarray([
            row for row in base_rows
            if padded_query in f" {normalized_title_array[row]} "
        ], dtype=np.int32)
        if len(exact_rows) == 0:
            predictions[query_id] = []
            continue

        bm25_scores = (
            query_matrix[query_row] @ bm25_item_matrix[exact_rows].T
        ).toarray().ravel()
        same_location = (
            np.isfinite(query_locations[query_row])
            & np.isfinite(item_locations[exact_rows])
            & (item_locations[exact_rows] == query_locations[query_row])
        ).astype(np.int8)

        order = np.lexsort((
            -bm25_scores,
            -item_reviews[exact_rows],
            -item_quality[exact_rows],
            -same_location,
        ))[:top_k]
        predictions[query_id] = item_ids[exact_rows[order]].tolist()

    return predictions


def build_candidate_pool(queries):
    """Пул не читает relevant_items и одинаково работает для train/test."""
    query_bm25 = bm25_vectorizer.transform(
        prepare_query_text(queries)
    ).tocsr()
    bm25_candidates = get_topk_bm25(queries, query_bm25)
    exact_candidates = get_topk_exact_titles(queries, query_bm25)

    rows = []
    for query_row, query_id in enumerate(queries["query_id"]):
        bm25_ranks = {
            item_id: rank
            for rank, item_id in enumerate(bm25_candidates[query_id], 1)
        }
        exact_ranks = {
            item_id: rank
            for rank, item_id in enumerate(exact_candidates[query_id], 1)
        }
        for item_id in dict.fromkeys([*bm25_ranks, *exact_ranks]):
            rows.append((
                query_row,
                int(item_row_by_id[item_id]),
                bm25_ranks.get(item_id, 0),
                exact_ranks.get(item_id, 0),
            ))

    pool = pd.DataFrame(
        rows,
        columns=["query_row", "item_row", "bm25_rank", "exact_rank"],
        dtype=np.int32,
    )
    assert not pool.duplicated(["query_row", "item_row"]).any()
    return pool, query_bm25


def label_train_pairs(pairs):
    relevant = train_queries["relevant_items"].tolist()
    return np.fromiter(
        (
            item_ids[item_row] in relevant[query_row]
            for query_row, item_row in
            pairs[["query_row", "item_row"]].itertuples(index=False, name=None)
        ),
        dtype=np.int8,
        count=len(pairs),
    )


Добавляем TF-IDF заголовков и собираем обучающие пары для ранжировщика

In [ ]:
pair_vectorizer = TfidfVectorizer(
    lowercase=True, ngram_range=(1, 1), min_df=2,
    max_features=200_000, dtype=np.float32,
)
pair_item_matrix = pair_vectorizer.fit_transform(normalized_titles).tocsr()


def get_topk_title_tfidf(
    queries, query_matrix, top_k=CANDIDATES_PER_CHANNEL,
    batch_size=SEARCH_BATCH_SIZE, n_workers=SEARCH_WORKERS,
):
    item_matrix_t = pair_item_matrix.T.tocsc()
    top_k = min(top_k, len(benchmark_items))
    starts = list(range(0, len(queries), batch_size))
    query_ids = queries["query_id"].to_numpy()

    def process_batch(start):
        end = min(start + batch_size, len(queries))
        scores = (query_matrix[start:end] @ item_matrix_t).toarray()
        indices = np.argpartition(
            scores, kth=scores.shape[1] - top_k, axis=1,
        )[:, -top_k:]
        candidate_scores = np.take_along_axis(scores, indices, axis=1)
        order = np.argsort(candidate_scores, axis=1)[:, ::-1]
        return start, end, np.take_along_axis(indices, order, axis=1)

    predictions = {}
    with ThreadPoolExecutor(max_workers=n_workers) as executor:
        results = executor.map(process_batch, starts)
        for start, end, indices in tqdm.tqdm(
            results, total=len(starts), desc="Title-TF-IDF top-k",
        ):
            for local_row, query_row in enumerate(range(start, end)):
                predictions[query_ids[query_row]] = (
                    item_ids[indices[local_row]].tolist()
                    if query_matrix.getrow(query_row).nnz else []
                )
    return predictions


def merge_title_rank(pool, queries, predictions):
    # Outer join добавляет найденные заголовки-кандидаты и сохраняет ранги BM25 и точных заголовков
    candidate_lists = [predictions[qid] for qid in queries["query_id"]]
    sizes = np.fromiter((len(x) for x in candidate_lists), dtype=np.int32)
    query_rows = np.repeat(np.arange(len(queries), dtype=np.int32), sizes)
    item_rows = np.fromiter(
        (int(item_row_by_id[item_id]) for x in candidate_lists for item_id in x),
        dtype=np.int32, count=int(sizes.sum()),
    )
    title_ranks = np.concatenate([
        np.arange(1, size + 1, dtype=np.int32) for size in sizes if size
    ])
    title_pairs = pd.DataFrame({
        "query_row": query_rows, "item_row": item_rows,
        "title_rank": title_ranks,
    })
    merged = pool.merge(
        title_pairs, on=["query_row", "item_row"],
        how="outer", sort=False, validate="one_to_one",
    )
    for column in ("bm25_rank", "exact_rank", "title_rank"):
        merged[column] = merged[column].fillna(0).astype(np.int32)
    assert not merged.duplicated(["query_row", "item_row"]).any()
    return merged


train_title_query_matrix = pair_vectorizer.transform(
    normalize_for_title_match(train_queries["search_query"])
).tocsr()
train_title_candidates = get_topk_title_tfidf(
    train_queries, train_title_query_matrix,
)
train_candidate_pool, train_query_bm25 = build_candidate_pool(train_queries)
train_candidate_pool = merge_title_rank(
    train_candidate_pool, train_queries, train_title_candidates,
)
pool_targets = label_train_pairs(train_candidate_pool)

# Сохраняем найденные положительные и верхние трудные отрицательные каждого канала
keep = (
    (pool_targets == 1)
    | train_candidate_pool["bm25_rank"].between(1, NEGATIVES_PER_CHANNEL)
    | train_candidate_pool["exact_rank"].between(1, NEGATIVES_PER_CHANNEL)
    | train_candidate_pool["title_rank"].between(1, NEGATIVES_PER_CHANNEL)
)
train_pairs = train_candidate_pool.loc[keep].copy()

# Положительные, пропущенные всеми каналами, добавляем только в обучение
known_positive_rows = [
    (query_row, int(item_row_by_id[item_id]), 0, 0, 0)
    for query_row, relevant in enumerate(train_queries["relevant_items"])
    for item_id in sorted(relevant)
]
train_pairs = (
    pd.concat([
        train_pairs,
        pd.DataFrame(
            known_positive_rows,
            columns=train_candidate_pool.columns,
        ),
    ], ignore_index=True)
    .drop_duplicates(["query_row", "item_row"])
    .reset_index(drop=True)
    .astype(np.int32)
)
train_y = label_train_pairs(train_pairs)

pool_hits = pd.Series(pool_targets).groupby(
    train_candidate_pool["query_row"].to_numpy()
).sum().reindex(range(len(train_queries)), fill_value=0)
train_pool_recall = (
    pool_hits.to_numpy()
    / train_queries["relevant_items"].map(len).to_numpy()
).mean()

print(f"Пар в исходном train-пуле: {len(train_candidate_pool):,}")
print(f"Полнота train-пула: {train_pool_recall:.5f}")
print(f"Обучающих пар: {len(train_pairs):,}")
print(f"Положительных обучающих пар: {train_y.sum():,}")
assert train_y.sum() == train_queries["relevant_items"].map(len).sum()


Title-TF-IDF top-k:  69%|██████▊   | 569/830 [00:45<00:19, 13.18it/s]

Вычисляем текстовые и числовые признаки каждой пары запрос–объявление

In [ ]:
train_query_tfidf = pair_vectorizer.transform(
    normalize_for_title_match(train_queries["search_query"])
).tocsr()


def make_pair_features(pairs, queries, query_tfidf, query_bm25):
    query_rows = pairs["query_row"].to_numpy()
    candidate_rows = pairs["item_row"].to_numpy()

    # Поэлементное произведение оставляет общие слова запроса и заголовки.
    overlap = query_tfidf[query_rows].multiply(
        pair_item_matrix[candidate_rows]
    ).tocsr()
    bm25_scores = np.asarray(
        query_bm25[query_rows]
        .multiply(bm25_item_matrix[candidate_rows])
        .sum(axis=1)
    ).ravel()

    query_texts = normalize_for_title_match(
        queries["search_query"]
    ).to_numpy()
    query_tokens = [set(text.split()) for text in query_texts]
    exact_title = np.fromiter(
        (
            bool(query_texts[q])
            and f" {query_texts[q]} " in f" {normalized_title_array[i]} "
            for q, i in zip(query_rows, candidate_rows)
        ),
        dtype=np.float32,
        count=len(pairs),
    )
    query_word_coverage = np.fromiter(
        (
            len(query_tokens[q] & title_token_sets[i])
            / max(1, len(query_tokens[q]))
            for q, i in zip(query_rows, candidate_rows)
        ),
        dtype=np.float32,
        count=len(pairs),
    )

    query_locations = pd.to_numeric(
        queries["search_location_id"], errors="coerce"
    ).to_numpy(dtype=float, na_value=np.nan)
    same_location = (
        np.isfinite(query_locations[query_rows])
        & np.isfinite(item_locations[candidate_rows])
        & (
            query_locations[query_rows]
            == item_locations[candidate_rows]
        )
    )

    log_reviews = np.log1p(item_reviews[candidate_rows])
    title_ranks = pairs["title_rank"].to_numpy(dtype=np.float32)
    title_rank_signal = np.zeros(len(pairs), dtype=np.float32)
    has_title_rank = title_ranks > 0
    title_rank_signal[has_title_rank] = 1.0 / np.log2(
        title_ranks[has_title_rank] + 1.0
    )
    numeric = np.column_stack([
        np.log1p(np.maximum(bm25_scores, 0)),
        exact_title,
        same_location,
        item_ratings[candidate_rows],
        log_reviews,
        item_ratings[candidate_rows] * log_reviews,
        query_word_coverage,
        np.asarray(overlap.sum(axis=1)).ravel(),
        title_rank_signal,
    ]).astype(np.float32)
    return overlap, numeric


Обучаем логистическую регрессию на подготовленных парах и признаках

In [ ]:
overlap_parts = []
numeric_parts = []

for start in tqdm.tqdm(
    range(0, len(train_pairs), PAIR_BATCH_SIZE),
    desc="Признаки обучающих пар",
):
    end = min(start + PAIR_BATCH_SIZE, len(train_pairs))
    overlap, numeric = make_pair_features(
        train_pairs.iloc[start:end],
        train_queries,
        train_query_tfidf,
        train_query_bm25,
    )
    overlap_parts.append(overlap)
    numeric_parts.append(numeric)

train_overlap = vstack(overlap_parts, format="csr")
train_numeric = np.vstack(numeric_parts)
del overlap_parts, numeric_parts

numeric_scaler = StandardScaler()
X_train = hstack([
    train_overlap,
    csr_matrix(numeric_scaler.fit_transform(train_numeric)),
], format="csr", dtype=np.float32)
del train_overlap, train_numeric

rerank_model = SGDClassifier(
    loss="log_loss",
    alpha=1e-5,
    class_weight="balanced",
    max_iter=30,
    tol=1e-3,
    random_state=RANDOM_SEED,
)
rerank_model.fit(X_train, train_y)

print("Матрица обучения:", X_train.shape)
print("Эпох обучения:", rerank_model.n_iter_)


Строим базовую выдачу для и добавляем историю знакомых запросов

In [ ]:
submission_pool, submission_query_bm25 = build_candidate_pool(
    submission_queries
)
submission_query_tfidf = pair_vectorizer.transform(
    normalize_for_title_match(submission_queries["search_query"])
).tocsr()

if USE_TITLE_TFIDF_CHANNEL:
    def get_topk_title_tfidf(
        queries, query_matrix, top_k=CANDIDATES_PER_CHANNEL,
        batch_size=SEARCH_BATCH_SIZE, n_workers=SEARCH_WORKERS,
    ):
        item_matrix_t = pair_item_matrix.T.tocsc()
        top_k = min(top_k, len(benchmark_items))
        batch_starts = list(range(0, len(queries), batch_size))
        query_ids = queries["query_id"].to_numpy()

        def process_batch(start):
            end = min(start + batch_size, len(queries))
            scores = (query_matrix[start:end] @ item_matrix_t).toarray()
            indices = np.argpartition(
                scores, kth=scores.shape[1] - top_k, axis=1,
            )[:, -top_k:]
            candidate_scores = np.take_along_axis(scores, indices, axis=1)
            order = np.argsort(candidate_scores, axis=1)[:, ::-1]
            return start, end, np.take_along_axis(indices, order, axis=1)

        predictions = {}
        with ThreadPoolExecutor(max_workers=n_workers) as executor:
            results = executor.map(process_batch, batch_starts)
            for start, end, indices in tqdm.tqdm(
                results, total=len(batch_starts),
                desc="Title-TF-IDF top-k",
            ):
                for local_row, query_row in enumerate(range(start, end)):
                    if query_matrix.getrow(query_row).nnz == 0:
                        predictions[query_ids[query_row]] = []
                    else:
                        predictions[query_ids[query_row]] = (
                            item_ids[indices[local_row]].tolist()
                        )
        return predictions


    def add_title_candidates(pool, queries, predictions):
        existing = set(
            pool[["query_row", "item_row"]]
            .itertuples(index=False, name=None)
        )
        rows = []
        for query_row, query_id in enumerate(queries["query_id"]):
            for item_id in predictions[query_id]:
                item_row = int(item_row_by_id[item_id])
                key = (query_row, item_row)
                if key not in existing:
                    rows.append((query_row, item_row, 0, 0))
                    existing.add(key)

        if rows:
            pool = pd.concat([
                pool,
                pd.DataFrame(rows, columns=pool.columns, dtype=np.int32),
            ], ignore_index=True)
        assert not pool.duplicated(["query_row", "item_row"]).any()
        return pool


    pool_size_before_title = len(submission_pool)
    submission_title_candidates = get_topk_title_tfidf(
        submission_queries, submission_query_tfidf,
        top_k=CANDIDATES_PER_CHANNEL,
    )
    submission_pool = add_title_candidates(
        submission_pool, submission_queries, submission_title_candidates,
    )
    print(
        f"Добавлено пар title-TF-IDF: "
        f"{len(submission_pool) - pool_size_before_title:,}"
    )

submission_pool = merge_title_rank(
    submission_pool, submission_queries, submission_title_candidates,
)
assert "title_rank" in submission_pool.columns

submission_scores = np.empty(len(submission_pool), dtype=np.float32)
for start in tqdm.tqdm(
    range(0, len(submission_pool), PAIR_BATCH_SIZE),
    desc="Оценка benchmark-кандидатов",
):
    end = min(start + PAIR_BATCH_SIZE, len(submission_pool))
    overlap, numeric = make_pair_features(
        submission_pool.iloc[start:end],
        submission_queries,
        submission_query_tfidf,
        submission_query_bm25,
    )
    X_batch = hstack([
        overlap,
        csr_matrix(numeric_scaler.transform(numeric)),
    ], format="csr", dtype=np.float32)
    submission_scores[start:end] = rerank_model.decision_function(X_batch)

ranked_submission = submission_pool.assign(
    score=submission_scores
).sort_values(
    ["query_row", "score", "item_row"],
    ascending=[True, False, True],
)
top50_submission = (
    ranked_submission.groupby("query_row", sort=False)
    .head(50)
    .copy()
)
top50_submission["item_id"] = item_ids[
    top50_submission["item_row"].to_numpy()
]
items_by_query_row = (
    top50_submission.groupby("query_row")["item_id"].agg(list).to_dict()
)
submission_predictions = [
    items_by_query_row.get(query_row, [])
    for query_row in range(len(submission_queries))
]

# История строится только по train-кликам и ограничивается доступным корпусом.
history_by_text = defaultdict(Counter)
for text, relevant in zip(train_queries["search_query"], train_queries["relevant_items"]):
    key = " ".join(text.lower().split()) if isinstance(text, str) else ""
    history_by_text[key].update(relevant)

HISTORY_LIMIT = 20
submission_with_history = []
queries_with_history = 0
for text, base_items in zip(submission_queries["search_query"], submission_predictions):
    key = " ".join(text.lower().split()) if isinstance(text, str) else ""
    history_items = [item for item, _ in history_by_text[key].most_common(HISTORY_LIMIT)]
    queries_with_history += bool(history_items)
    selected = list(dict.fromkeys(history_items))
    seen = set(selected)
    for item_id in base_items:
        if item_id not in seen:
            selected.append(item_id)
            seen.add(item_id)
        if len(selected) == 50:
            break
    submission_with_history.append(selected)
submission_predictions = submission_with_history
print(f"Запросов с train-историей: {queries_with_history:,} / {len(submission_queries):,}")

print(f"Пар в benchmark-пуле: {len(submission_pool):,}")
print(pd.Series(map(len, submission_predictions)).value_counts().sort_index())


Загружаем granite-embedding-97m-multilingual-r2, кодируем тексты и создаём индекс эмбеддингов

In [ ]:
# Выбираем GPU или CPU, кодируем тексты Granite и создаём индекс эмбеддингов.

import hashlib
import faiss
import torch
from sentence_transformers import SentenceTransformer

EMBED_MODEL_ID = "ibm-granite/granite-embedding-97m-multilingual-r2"
EMBED_MAX_TOKENS = 512
EMBED_BATCH_SIZE = 16
DENSE_TOP_K = CANDIDATES_PER_CHANNEL
EMBED_DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

embed_model = SentenceTransformer(EMBED_MODEL_ID, device=EMBED_DEVICE)
assert embed_model.device.type == EMBED_DEVICE
embed_model.max_seq_length = EMBED_MAX_TOKENS
print("Granite device:", embed_model.device)
if EMBED_DEVICE == "cpu":
    print("CUDA недоступна: создание эмбеддингов на CPU займёт больше времени.")

dense_item_texts = (
    "Заголовок: " + benchmark_items["item_title_raw"].fillna("")
    + ". Параметры: " + clean_params(benchmark_items["item_infm_params_text"])
    + ". Описание: " + clean_description(benchmark_items["item_description_raw"])
).tolist()
text_hashes = pd.util.hash_pandas_object(
    pd.Series(dense_item_texts), index=False,
).to_numpy(dtype=np.uint64)
corpus_fingerprint = hashlib.sha256(text_hashes.tobytes()).hexdigest()[:16]
cache_dir = DATA_DIR / "embedding_cache"
cache_dir.mkdir(exist_ok=True)
cache_path = cache_dir / f"granite_97m_r2_{EMBED_MAX_TOKENS}_{corpus_fingerprint}.npy"
if cache_path.exists():
    dense_item_embeddings = np.load(cache_path, mmap_mode="r")
    print("Dense-кэш:", cache_path)
else:
    dense_item_embeddings = embed_model.encode_document(
        dense_item_texts, batch_size=EMBED_BATCH_SIZE,
        normalize_embeddings=True, convert_to_numpy=True,
        show_progress_bar=True,
    ).astype(np.float32)
    np.save(cache_path, dense_item_embeddings)
    print("Dense-кэш создан:", cache_path)
del dense_item_texts, text_hashes
assert dense_item_embeddings.shape == (len(benchmark_items), 384)

dense_index = faiss.IndexHNSWFlat(384, 32, faiss.METRIC_INNER_PRODUCT)
dense_index.hnsw.efConstruction = 160
dense_index.hnsw.efSearch = max(256, DENSE_TOP_K)
dense_index.add(np.ascontiguousarray(dense_item_embeddings, dtype=np.float32))
assert dense_index.ntotal == len(benchmark_items)


def get_dense_predictions(queries):
    texts = (
        "Запрос: " + queries["search_query"].fillna("")
        + ". Фильтры: " + clean_params(queries["search_infm_params_text"])
    ).tolist()
    embeddings = embed_model.encode_query(
        texts, batch_size=EMBED_BATCH_SIZE,
        normalize_embeddings=True, convert_to_numpy=True,
        show_progress_bar=True,
    ).astype(np.float32)
    _, rows = dense_index.search(np.ascontiguousarray(embeddings), DENSE_TOP_K)
    predictions = {
        query_id: item_ids[found[found >= 0]].tolist()
        for query_id, found in zip(queries["query_id"], rows)
    }
    return predictions, embeddings

Добавляем кандидатов от эмбеддинговой модели и переобучаем ранжировщик с признаками семантической близости

In [ ]:
def dense_rank_frame(queries, predictions):
    candidate_lists = [predictions[qid] for qid in queries["query_id"]]
    sizes = np.fromiter((len(items) for items in candidate_lists), dtype=np.int32)
    query_rows = np.repeat(np.arange(len(queries), dtype=np.int32), sizes)
    item_rows = np.fromiter(
        (int(item_row_by_id[item_id]) for items in candidate_lists for item_id in items),
        dtype=np.int32, count=int(sizes.sum()),
    )
    ranks = np.concatenate([
        np.arange(1, size + 1, dtype=np.int32) for size in sizes if size
    ])
    return pd.DataFrame({
        "query_row": query_rows, "item_row": item_rows, "dense_rank": ranks,
    })


def merge_dense_rank(pool, queries, predictions):
    merged = pool.merge(
        dense_rank_frame(queries, predictions),
        on=["query_row", "item_row"],
        how="outer", sort=False, validate="one_to_one",
    )
    for column in ("bm25_rank", "exact_rank", "title_rank", "dense_rank"):
        merged[column] = merged[column].fillna(0).astype(np.int32)
    return merged


dense_train_predictions, dense_train_embeddings = get_dense_predictions(train_queries)
train_dense_ranks = dense_rank_frame(train_queries, dense_train_predictions)
dense_hard_pairs = train_dense_ranks.loc[
    train_dense_ranks["dense_rank"].le(NEGATIVES_PER_CHANNEL),
    ["query_row", "item_row"],
]
hybrid_train_pairs = pd.concat([
    train_pairs,
    dense_hard_pairs.assign(bm25_rank=0, exact_rank=0, title_rank=0)[train_pairs.columns],
], ignore_index=True).drop_duplicates(["query_row", "item_row"])
hybrid_train_pairs = hybrid_train_pairs.merge(
    train_dense_ranks, on=["query_row", "item_row"],
    how="left", sort=False, validate="one_to_one",
)
hybrid_train_pairs["dense_rank"] = (
    hybrid_train_pairs["dense_rank"].fillna(0).astype(np.int32)
)
hybrid_train_pairs = hybrid_train_pairs.reset_index(drop=True)
hybrid_train_y = label_train_pairs(hybrid_train_pairs)
assert hybrid_train_y.sum() == train_y.sum()
print(f"Hybrid train pairs: {len(hybrid_train_pairs):,}; positives: {hybrid_train_y.sum():,}")


def make_hybrid_features(pairs, queries, query_tfidf, query_bm25, query_embeddings):
    overlap, numeric = make_pair_features(
        pairs, queries, query_tfidf, query_bm25,
    )
    q = pairs["query_row"].to_numpy()
    i = pairs["item_row"].to_numpy()
    cosine = np.einsum(
        "ij,ij->i",
        np.asarray(query_embeddings[q], dtype=np.float32),
        np.asarray(dense_item_embeddings[i], dtype=np.float32),
    )
    ranks = pairs["dense_rank"].to_numpy(dtype=np.float32)
    rank_signal = np.zeros(len(ranks), dtype=np.float32)
    found = ranks > 0
    rank_signal[found] = 1.0 / np.log2(ranks[found] + 1.0)
    numeric = np.column_stack([numeric, cosine, rank_signal]).astype(np.float32)
    return overlap, numeric


hybrid_overlap_parts, hybrid_numeric_parts = [], []
for start in tqdm.tqdm(
    range(0, len(hybrid_train_pairs), PAIR_BATCH_SIZE),
    desc="Признаки hybrid train-пар",
):
    end = min(start + PAIR_BATCH_SIZE, len(hybrid_train_pairs))
    overlap, numeric = make_hybrid_features(
        hybrid_train_pairs.iloc[start:end], train_queries,
        train_query_tfidf, train_query_bm25, dense_train_embeddings,
    )
    hybrid_overlap_parts.append(overlap)
    hybrid_numeric_parts.append(numeric)

hybrid_overlap = vstack(hybrid_overlap_parts, format="csr")
hybrid_numeric = np.vstack(hybrid_numeric_parts)
hybrid_scaler = StandardScaler()
hybrid_X_train = hstack([
    hybrid_overlap,
    csr_matrix(hybrid_scaler.fit_transform(hybrid_numeric)),
], format="csr", dtype=np.float32)
del hybrid_overlap_parts, hybrid_numeric_parts, hybrid_overlap, hybrid_numeric

hybrid_model = SGDClassifier(
    loss="log_loss", alpha=1e-5, class_weight="balanced",
    max_iter=30, tol=1e-3, random_state=RANDOM_SEED,
)
hybrid_model.fit(hybrid_X_train, hybrid_train_y)
print("Hybrid train matrix:", hybrid_X_train.shape)
print("Epochs:", hybrid_model.n_iter_)

Формируем гибридную выдачу для запросов с учётом истории

In [ ]:
dense_submission_predictions, dense_submission_embeddings = get_dense_predictions(
    submission_queries,
)
submission_hybrid_pool = merge_dense_rank(
    submission_pool, submission_queries, dense_submission_predictions,
)
submission_scores = np.empty(len(submission_hybrid_pool), dtype=np.float32)
for start in tqdm.tqdm(
    range(0, len(submission_hybrid_pool), PAIR_BATCH_SIZE),
    desc="Оценка hybrid benchmark-кандидатов",
):
    end = min(start + PAIR_BATCH_SIZE, len(submission_hybrid_pool))
    overlap, numeric = make_hybrid_features(
        submission_hybrid_pool.iloc[start:end], submission_queries,
        submission_query_tfidf, submission_query_bm25,
        dense_submission_embeddings,
    )
    X_batch = hstack([
        overlap, csr_matrix(hybrid_scaler.transform(numeric)),
    ], format="csr", dtype=np.float32)
    submission_scores[start:end] = hybrid_model.decision_function(X_batch)

ranked_submission = submission_hybrid_pool.assign(score=submission_scores).sort_values(
    ["query_row", "score", "item_row"], ascending=[True, False, True],
)
top50_submission = ranked_submission.groupby("query_row", sort=False).head(50)
top50_submission = top50_submission.assign(
    item_id=item_ids[top50_submission["item_row"].to_numpy()]
)
items_by_query_row = top50_submission.groupby("query_row")["item_id"].agg(list).to_dict()
submission_predictions = []
for query_row, text in enumerate(submission_queries["search_query"]):
    key = " ".join(text.lower().split()) if isinstance(text, str) else ""
    history_items = [
        item_id for item_id, _ in history_by_text[key].most_common(HISTORY_LIMIT)
    ]
    selected = list(dict.fromkeys(history_items))
    for item_id in items_by_query_row.get(query_row, []):
        if item_id not in selected:
            selected.append(item_id)
        if len(selected) == 50:
            break
    submission_predictions.append(selected)
print(f"Hybrid benchmark-пул: {len(submission_hybrid_pool):,} пар")
print(pd.Series(map(len, submission_predictions)).value_counts().sort_index())

Сохраняем ответ


In [ ]:
answer = pd.DataFrame({
    "query_id": benchmark_queries["query_id"].astype(str),
    "answer": [
        " ".join(map(str, item_ids_for_query))
        for item_ids_for_query in submission_predictions
    ],
})

answer_lists = answer["answer"].str.split()
predicted_item_ids = set(chain.from_iterable(answer_lists))

answer.to_csv(
    ANSWER_PATH,
    index=False,
    encoding="utf-8",
    lineterminator="\n",
)